In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd

# Ab baaki ka poora code chalega...
# 1. Path routing setup
sys.path.append(os.path.abspath(os.path.join('..')))

# 2. Imports
from src.data_ingestion import DataIngestion
from src.data_transformation import Vocabulary
from src.dataset import ImageCaptionDataset, CapsCollate
from src.model import ImageEncoder, CaptionDecoder

# 3. Setup paths
CAPTION_FILE = "../data/raw/captions.txt"  
IMAGE_DIR = "../data/raw/Images/"     

# 4. Data check and split
ingestion = DataIngestion(caption_file_path=CAPTION_FILE, image_dir=IMAGE_DIR)
df = ingestion.load_data()

train_df, val_df = ingestion.split_data(df, test_size=0.2, random_state=42)

# 5. Build Vocabulary
vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(train_df['caption'].tolist())
vocab_size = len(vocab)
print(f"✨ Success! Total unique vocabulary words: {vocab_size}")

# 6. Data Loader setup
train_dataset = ImageCaptionDataset(train_df, vocab)
pad_idx = vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,       
    shuffle=True,
    num_workers=0,  
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

✨ Success! Total unique vocabulary words: 2671


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# Add root directory imports safely
from src.model import ImageEncoder, CaptionDecoder

# --- THE ABSOLUTE FIX: Define device right here inside this cell ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# -----------------------------------------------------------------

# 1. Hyperparameters setup
EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
LEARNING_RATE = 3e-4
EPOCHS = 5

# 2. Ensure vocab variables exist from the previous block
try:
    vocab_size = len(vocab)
    pad_idx = vocab.stoi["<PAD>"]
except NameError:
    # Backup load agar top cell variables break hue hon
    vocab_size = len(train_dataset.vocab)
    pad_idx = train_dataset.vocab.stoi["<PAD>"]

# 3. Build and push models to your active hardware engine
encoder = ImageEncoder(embed_size=EMBED_SIZE).to(device)
decoder = CaptionDecoder(
    embed_size=EMBED_SIZE, 
    hidden_size=HIDDEN_SIZE, 
    vocab_size=vocab_size, 
    num_layers=NUM_LAYERS
).to(device)

# 4. Set Loss and Gradient Optimizer configuration
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
parameters = list(decoder.parameters()) + list(encoder.embed.parameters())
optimizer = optim.Adam(parameters, lr=LEARNING_RATE)

print(f"✨ Success! Neural network brain models successfully built and initialized on {device}.")

c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\nitis\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✨ Success! Neural network brain models successfully built and initialized on cpu.


In [11]:
import torch
import torch.nn as nn
from src.model import ImageEncoder, CaptionDecoder

# ==========================================
# 1. INITIAL SYSTEM CONFIGURATIONS
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_SIZE = 256
HIDDEN_SIZE = 512
VOCAB_SIZE = len(vocab)  # Apne dataset ka actual vocabulary length variable (e.g., len(vocab))

# Models ko initialize karke device par bhejo
encoder = ImageEncoder(embed_size=EMBED_SIZE).to(DEVICE)
decoder = CaptionDecoder(embed_size=EMBED_SIZE, hidden_size=HIDDEN_SIZE, vocab_size=VOCAB_SIZE).to(DEVICE)

# ==========================================
# 2. LOAD SAVED WEIGHTS (EPOCH 5 CHECKPOINTS)
# ==========================================
print("🔄 Loading existing Epoch 5 checkpoints...")
encoder.load_state_dict(torch.load("../artifacts/encoder_epoch_5.pth", map_location=DEVICE))
decoder.load_state_dict(torch.load("../artifacts/decoder_epoch_5.pth", map_location=DEVICE))
print("✅ Weights successfully restored from artifacts folder!")

# ==========================================
# 3. ENCODER FREEZING & FINE-TUNING SETUP
# ==========================================
# ResNet Base ko freeze rakhenge taaki laptop CPU par load na aaye
for param in encoder.resnet.parameters():
    param.requires_grad = False

# Loss and Optimizer setup (Learning Rate reduced to 0.0003 for smooth convergence)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
optimizer = torch.optim.Adam(
    list(decoder.parameters()) + list(encoder.embed.parameters()), 
    lr=0.0003
)

# ==========================================
# 4. RESUME LOOP (STARTING FROM EPOCH 6)
# ==========================================
start_epoch = 6
num_epochs = 15  # Isko badha kar 20 ya 25 bhi kar sakte ho training ke dauran

print(f"🔥 Resuming training loop dynamically from Epoch {start_epoch} to {num_epochs}...\n")

for epoch in range(start_epoch, num_epochs + 1):
    encoder.train()
    decoder.train()
    total_loss = 0
    
    # data_loader tumhara wahi rehga jo tumne notebook ke shuruat mein define kiya tha
    for idx, (images, captions) in enumerate(train_loader): 
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward Pass execution pipeline
        features = encoder(images)
        outputs = decoder(features, captions)
        
        # Loss calculation (flattening outputs to match vocabulary shape)
        loss = criterion(outputs.view(-1, VOCAB_SIZE), captions.view(-1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Every 100 batches progress print karne ke liye
        if idx % 100 == 0:
            print(f"Epoch [{epoch}/{num_epochs}], Step [{idx}/{len(train_loader)}], Current Loss: {loss.item():.4f}")
        
    # Average Loss calculation for the whole epoch
    epoch_loss = total_loss / len(train_loader)
    print(f"🏁 Epoch [{epoch}/{num_epochs}] Completed | Avg Loss: {epoch_loss:.4f}\n")
    
    # Tracking metrics by saving weights iteratively (har epoch ke baad checkpoint backup)
    torch.save(encoder.state_dict(), f"../artifacts/encoder_epoch_{epoch}.pth")
    torch.save(decoder.state_dict(), f"../artifacts/decoder_epoch_{epoch}.pth")

🔄 Loading existing Epoch 5 checkpoints...
✅ Weights successfully restored from artifacts folder!
🔥 Resuming training loop dynamically from Epoch 6 to 15...

Epoch [6/15], Step [0/1012], Current Loss: 2.2656
Epoch [6/15], Step [100/1012], Current Loss: 2.3191
Epoch [6/15], Step [200/1012], Current Loss: 2.5587
Epoch [6/15], Step [300/1012], Current Loss: 2.2570
Epoch [6/15], Step [400/1012], Current Loss: 2.3541
Epoch [6/15], Step [500/1012], Current Loss: 2.4164
Epoch [6/15], Step [600/1012], Current Loss: 2.3491
Epoch [6/15], Step [700/1012], Current Loss: 2.4164
Epoch [6/15], Step [800/1012], Current Loss: 2.4212
Epoch [6/15], Step [900/1012], Current Loss: 2.5074
Epoch [6/15], Step [1000/1012], Current Loss: 2.0681
🏁 Epoch [6/15] Completed | Avg Loss: 2.3687

Epoch [7/15], Step [0/1012], Current Loss: 2.3450
Epoch [7/15], Step [100/1012], Current Loss: 2.3704
Epoch [7/15], Step [200/1012], Current Loss: 2.1350
Epoch [7/15], Step [300/1012], Current Loss: 2.2836
Epoch [7/15], Step [4

In [ ]:
def generate_caption_final(image_path, encoder, decoder, vocabulary):
    image = Image.open(image_path).convert("RGB")
    
    image = transform(image).unsqueeze(0).to(device)
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        # 1. Feature extract karke shape [1, 1, embed_size] banao
        features = encoder(image).unsqueeze(1)
        current_inputs = features
        result_caption = []
        
        for _ in range(20):
            # Sequence ko LSTM me bhejo
            lstm_out, _ = decoder.lstm(current_inputs)
            outputs = decoder.linear(lstm_out)
            
            # Harvey hata kar seedhe slicing: [batch_size, last_timestep, vocab_size]
            next_word_logits = outputs[:, -1, :] 
            predicted = next_word_logits.argmax(1)
            
            word = vocabulary.itos[predicted.item()]
            
            if word == "<EOS>":
                break
            if word != "<SOS>":
                result_caption.append(word)
                
            # Naye token ko embed karke sequence me concatenate karo
            next_embed = decoder.embed(predicted).unsqueeze(1)
            current_inputs = torch.cat((current_inputs, next_embed), dim=1)
            
    print(f"\nActual Caption: {' '.join(result_caption)}")

# Call karne ke liye dhyan rakhna agar variable name 'vocabulary' hai toh wahi pass karna:
generate_caption_final("../test.jpg", encoder, decoder, vocab)

SyntaxError: invalid syntax (3292551393.py, line 23)